In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

def highlight_max(row):
    is_max = row == row.max()
    return ['font-weight: bold' if v else '' for v in is_max]

def drop_nan_group(group):
    if group.isnull().any():
        return None  # Returning None will drop the group
    else:
        return group

def compte_les_points(df):
    results = {}
    total = 0
    for idx, row in df.reset_index()['perf'].iterrows():
        is_max = row == row.max()
        for key in is_max.reset_index()['package'].values:
            if key not in results:
                results[key] = 0
                
            results[key] += int(is_max[key])
        total += 1
            
    for key in results:
        print(f"{key} : {results[key]} / {total}")

In [2]:
# Load bench 
naive = pd.read_csv("/Users/rudy/Documents/CHU/iias/baiddy_group/automl/src/perf_logger/bench/bench2_naive.csv")
naive['package'] = 'NaiveAutoML'

In [16]:
naive['perf'] = naive.apply(lambda row: row['balanced_accuracy'] if pd.notnull(row['balanced_accuracy']) else row['r2_score'], axis=1)

In [25]:
grouped = naive.groupby(['package', 'max_duration', 'dataset']).mean(numeric_only=True).reset_index()[['package', 'perf', 'max_duration', 'dataset']]

# grouped

In [26]:
pivot_df = grouped.pivot(index=['dataset', 'max_duration'], columns='package')
pivot_df.style.apply(highlight_max, axis=1)

In [4]:
# def count(row):
#     is_max = row == row.max()
#     return ['font-weight: bold' if v else '' for v in is_max]

# pivot_df = grouped.pivot(index=['dataset_name', 'duration'], columns='package')

# pivot_df.style.apply(highlight_max, axis=1)

compte_les_points(pivot_df)

AutoMed22 : 37 / 74
AutoMed_mutate : 11 / 74
AutoMed_void : 38 / 74


In [ ]:
# Auto-Sklearn : 14 / 75
# AutoMed : 29 / 75
# NaiveAutoML : 38 / 75

In [ ]:
# PAR DATASET sans prendre ne compte le temps
merged_df = pd.concat([naive, automed_void, autosk])[['package', 'duration', 'dataset_name', 'perf']]
# merged_df = pd.concat([naive, automed, automed21, automed22, autosk])[['package', 'duration', 'dataset_name', 'perf']]

merged_df = merged_df.dropna() # Avoid penalize a package if no value

merged_df.drop(columns=['duration'], inplace=True)
grouped = merged_df.groupby(['package', 'dataset_name']).mean(numeric_only=True).reset_index()

pivot_df = grouped.pivot(index=['dataset_name'], columns='package')
pivot_df.style.apply(highlight_max, axis=1)

In [ ]:
compte_les_points(pivot_df)

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

def line_plot(df, filter_tuple, hue="dataset_name",zero_line=False, y_lim=None):
    plt.figure()
    plot = sns.lineplot(data=df[df[filter_tuple[0]] == filter_tuple[1]], x="duration", y="perf", hue=hue) 
    sns.move_legend(plot, "upper left", bbox_to_anchor=(1, 1))
    plt.title(str(filter_tuple))
    if zero_line:
        plt.axhline(y=0.0, color='r', linestyle='--')
        
    if y_lim:
        plt.ylim(*y_lim)
    plt.show()


merged_df = pd.concat([naive, automed_void, autosk])[['package', 'duration', 'dataset_name', 'perf']]
merged_df = merged_df.dropna() # Avoid penalize a package if no value


In [ ]:
line_plot(merged_df, ("package", "AutoMed_void"))

In [ ]:
line_plot(merged_df, ("package", "NaiveAutoML"))

In [ ]:
line_plot(merged_df, ("package", "Auto-Sklearn"))

In [ ]:
for dataset_name in merged_df['dataset_name'].unique():
    line_plot(merged_df, ("dataset_name", dataset_name), hue="package")

In [ ]:
# Diff with automed ?
# Ensure df is a copy to avoid SettingWithCopyWarning
df = merged_df.copy()

# Separate 'automed' and other packages
automed_df = df[df['package'] == 'AutoMed_void']
other_df = df[df['package'] != 'AutoMed_void']

# Average 'automed' performance for each 'duration' and 'dataset_name' combination
automed_avg_perf = automed_df.groupby(['duration', 'dataset_name'])['perf'].max()

# Merge this average performance back into the other_df based on 'duration' and 'dataset_name'
other_df = other_df.merge(automed_avg_perf, on=['duration', 'dataset_name'], suffixes=('', '_automed_avg'))

# Calculate the difference in performance
other_df['perf'] = other_df['perf'] - other_df['perf_automed_avg']

# Drop the now unnecessary 'perf_automed_avg' column
other_df.drop(columns=['perf_automed_avg'], inplace=True)

# Resulting DataFrame has 'automed' rows removed and 'perf' adjusted
df_result = other_df

In [ ]:
for dataset_name in merged_df['dataset_name'].unique():
    try:
        line_plot(df_result, ("dataset_name", dataset_name), hue="package", zero_line=True, y_lim=(-0.2, 0.2))
    except:
        print(f"error for {dataset_name}")